In [2]:
import numpy as np
import pandas as pd

In [3]:
url_gasto_energetico = "../data/processed/gasto_energetico_hogar.csv"

df_gasto_energetico = pd.read_csv(url_gasto_energetico)

print(df_gasto_energetico.columns)


Index(['NUMERO', 'GASTO_SUMINISTROS', 'ANOENC', 'CLAVE', 'CLATEO', 'CCAA',
       'NUTS1', 'CAPROV', 'TAMAMU', 'DENSIDAD',
       ...
       'PENSIOTR', 'DESEM', 'OTRSUB', 'RENTAS', 'OTROIN', 'FUENPRIN',
       'FUENPRINRED', 'IMPEXAC', 'INTERIN', 'NUMPERI'],
      dtype='str', length=123)


In [4]:
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 1000):
    resumen_gastos = pd.DataFrame({
        "dtype": df_gasto_energetico.dtypes.astype(str),
        "no_nulos": df_gasto_energetico.notna().sum(),
        "nulos_nan": df_gasto_energetico.isna().sum(),
        "pct_nulos_nan": (df_gasto_energetico.isna().mean() * 100).round(2),
        "n_unicos": df_gasto_energetico.nunique(dropna=True),
        "muestra_valores": df_gasto_energetico.apply(lambda s: list(s.dropna().unique()[:3]))
    })
    display(resumen_gastos)

,dtype,no_nulos,nulos_nan,pct_nulos_nan,n_unicos,muestra_valores
NUMERO,int64,19544,0,0.00,19544,"[1, 2, 3]"
GASTO_SUMINISTROS,float64,19544,0,0.00,19527,"[760.428414, 1030.630587, 1265.599996]"
ANOENC,int64,19544,0,0.00,1,[2025]
CLAVE,str,19544,0,0.00,2,"[Segunda, Primera]"
CLATEO,str,19544,0,0.00,2,"[Segunda, Primera]"
CCAA,str,19544,0,0.00,19,"[Cataluña, Comunitat Valenciana, Galicia]"
NUTS1,str,19544,0,0.00,7,"[Este, Noroeste, Noreste]"
CAPROV,str,19544,0,0.00,2,"[No, Sí]"
TAMAMU,str,19544,0,0.00,5,[Municipio con 10.000 o más y menos de 20.000 ...
DENSIDAD,str,19544,0,0.00,3,"[Zona intermedia, Zona densamente poblada, Zon..."


In [5]:
cols = df_gasto_energetico.columns.str.strip()
gasto_mask =  cols.str.startswith("GAST", na= False) & (cols != "GASTO_SUMINISTROS")
columnas_a_eliminar = df_gasto_energetico.columns[gasto_mask].to_list()

columnas_a_eliminar += ["NUMERO", "ANOENC", "TAMANO", "IMPUTGAS"]

df_gasto_energetico["FACTOR"].describe()

count    19544.000000
mean       994.407621
std        844.628157
min         24.137465
25%        449.119171
50%        742.565274
75%       1272.800068
max      10954.976166
Name: FACTOR, dtype: float64

In [6]:
norden_mask =  cols.str.startswith("NORDEN", na= False)
columnas_a_eliminar += df_gasto_energetico.columns[norden_mask].to_list()

In [7]:
print(df_gasto_energetico["CLAVE"].value_counts(dropna=False))
print(df_gasto_energetico["CLATEO"].value_counts(dropna=False))
df_gasto_energetico[["CLAVE", "CLATEO"]].sample(10)

CLAVE
Primera    13043
Segunda     6501
Name: count, dtype: int64
CLATEO
Segunda    10560
Primera     8984
Name: count, dtype: int64


,CLAVE,CLATEO
17430,Segunda,Segunda
16144,Primera,Segunda
1932,Primera,Segunda
299,Segunda,Segunda
9430,Primera,Primera
8835,Primera,Segunda
4745,Primera,Primera
12754,Segunda,Segunda
4006,Primera,Segunda
16926,Primera,Primera


In [8]:
columnas_a_eliminar += ["CLAVE", "CLATEO", "FACTOR"]

In [9]:
s_imp = pd.to_numeric(df_gasto_energetico["IMPEXACPSP"], errors="coerce")
conteos = s_imp.value_counts(dropna=True)

menores_cero = conteos[conteos.index < 0]

total_menores = menores_cero.sum()
porcentaje_menores = (total_menores / len(df_gasto_energetico)) * 100

print(f"Total < 0: {total_menores}, porcentaje sobre total de filas: {porcentaje_menores:.2f}%")

Total < 0: 15382, porcentaje sobre total de filas: 78.70%


In [10]:
s_imp = pd.to_numeric(df_gasto_energetico["VALOREST"], errors="coerce")
conteos = s_imp.value_counts(dropna=True)

menores_cero = conteos[conteos.index < 0]

total_menores = menores_cero.sum()
porcentaje_menores = (total_menores / len(df_gasto_energetico)) * 100

print(f"Total < 0: {total_menores}, porcentaje sobre total de filas: {porcentaje_menores:.2f}%")

Total < 0: 3622, porcentaje sobre total de filas: 18.53%


In [11]:
s_imp = pd.to_numeric(df_gasto_energetico["IMPEXAC"], errors="coerce")
conteos = s_imp.value_counts(dropna=True)

menores_cero = conteos[conteos.index < 0]

total_menores = menores_cero.sum()
porcentaje_menores = (total_menores / len(df_gasto_energetico)) * 100

print(f"Total < 0: {total_menores}, porcentaje sobre total de filas: {porcentaje_menores:.2f}%")

Total < 0: 0, porcentaje sobre total de filas: 0.00%


In [12]:

# En porcentaje (0 a 100 %) redondeado a 2 decimales
porcentajes = (df_gasto_energetico["REGTEN"].value_counts(normalize=True, dropna=False) * 100).round(2)
print(porcentajes)

REGTEN
En propiedad sin préstamo o hipoteca en curso    53.40
En propiedad con préstamo o hipoteca en curso    26.99
Alquiler                                         14.16
Cesión gratuita                                   2.63
Cesión semigratuita                               1.70
Alquiler reducido                                 1.11
Name: proportion, dtype: float64


In [13]:
df_interinpsp_null = df_gasto_energetico.groupby("INTERINPSP")["INTERINPSP"].value_counts()
df_interinpsp_null


INTERINPSP
3000 o más euros                   492
De 1000 a menos de 1500 euros     1618
De 1500 a menos de 2000 euros     1421
De 2000 a menos de 2500 euros      889
De 2500 a menos de 3000 euros      439
De 500 a menos de 1000 euros       535
Menos de 500 euros                  57
No consta                        13999
Name: count, dtype: int64

In [14]:
columnas_a_eliminar += ["IMPEXACPSP", "INTERINPSP", "INTERIN"]
columnas_a_eliminar

['GASTOT',
 'GASTMON',
 'GASTNOM1',
 'GASTNOM2',
 'GASTNOM3',
 'GASTNOM4',
 'GASTOTGR01',
 'GASTOTGR02',
 'GASTOTGR03',
 'GASTOTGR04',
 'GASTOTGR05',
 'GASTOTGR06',
 'GASTOTGR07',
 'GASTOTGR08',
 'GASTOTGR09',
 'GASTOTGR10',
 'GASTOTGR11',
 'GASTOTGR12',
 'GASTOTGR13',
 'NUMERO',
 'ANOENC',
 'TAMANO',
 'IMPUTGAS',
 'NORDENSP',
 'NORDENCOSP',
 'NORDENP1SP',
 'NORDENP2SP',
 'CLAVE',
 'CLATEO',
 'FACTOR',
 'IMPEXACPSP',
 'INTERINPSP',
 'INTERIN']

In [15]:
df_calef_na = df_gasto_energetico[df_gasto_energetico["FUENCALE"].isna()][["FUENCALE", "CALEF"]]
display(df_calef_na)
df_calef_na["CALEF"].nunique()

,FUENCALE,CALEF
2,NaN,No
10,NaN,No
14,NaN,No
15,NaN,No
18,NaN,No
...,...,...
19520,NaN,No
19524,NaN,No
19526,NaN,No
19528,NaN,No


1

In [16]:
df_calef_ok = df_gasto_energetico[~df_gasto_energetico["FUENCALE"].isna()][["FUENCALE", "CALEF"]]
display(df_calef_ok)
df_calef_ok["CALEF"].nunique()

,FUENCALE,CALEF
0,Electricidad,Si
1,Gas natural,Si
3,Gas natural,Si
4,Gas natural,Si
5,Gas natural,Si
...,...,...
19539,Gas natural,Si
19540,Electricidad,Si
19541,Gas natural,Si
19542,Gas natural,Si


1

In [17]:
df_jornada_na = df_gasto_energetico[df_gasto_energetico["JORNADASP"].isna()][["JORNADASP", "OCUSP", "SITUREDSP"]]
display(df_jornada_na)
df_jornada_na["OCUSP"].nunique()

,JORNADASP,OCUSP,SITUREDSP
1,NaN,No ocupado,Inactivo
4,NaN,No ocupado,Inactivo
6,NaN,No ocupado,Inactivo
9,NaN,No ocupado,Activo
10,NaN,No ocupado,Activo
...,...,...,...
19531,NaN,No ocupado,Inactivo
19534,NaN,No ocupado,Inactivo
19535,NaN,No ocupado,Inactivo
19541,NaN,No ocupado,Inactivo


1

In [18]:
df_jornada_ok = df_gasto_energetico[~df_gasto_energetico["JORNADASP"].isna()][["JORNADASP", "OCUSP", "SITUREDSP"]]
display(df_jornada_ok)
print(df_jornada_ok["JORNADASP"].nunique())
print(df_jornada_ok["OCUSP"].nunique())
print(df_jornada_ok["SITUREDSP"].nunique())

,JORNADASP,OCUSP,SITUREDSP
0,Completa,Ocupado,Activo
2,Completa,Ocupado,Activo
3,Completa,Ocupado,Activo
5,Completa,Ocupado,Activo
7,Completa,Ocupado,Activo
...,...,...,...
19537,Completa,Ocupado,Activo
19538,Completa,Ocupado,Activo
19539,Completa,Ocupado,Activo
19540,Completa,Ocupado,Activo


2
1
1


In [19]:
print(df_gasto_energetico["SUPERF"].unique().tolist())
print(df_gasto_energetico["EDADSP"].unique().tolist())
print(df_gasto_energetico["NHABIT"].unique().tolist())

cols = ["SUPERF", "EDADSP", "NHABIT"]

for col in cols:
    s = df_gasto_energetico[col].astype(str).str.strip().str.lower()
    df_gasto_energetico[col] = pd.to_numeric(s.str.extract(r"(\d+)")[0], errors="coerce")

df_gasto_energetico[cols].info()

['085', '280', '090', '070', '105', '200', '300 metros o más', '100', '075', '088', '099', '086', '250', '092', '180', '053', '140', '065', '160', '095', '089', '069', '112', '060', '120', '072', '170', '148', '110', '050', '082', '103', '076', '115', '083', '098', '150', '080', '067', '35 metros o menos', '084', '078', '161', '063', '122', '062', '108', '049', '074', '093', '068', '158', '130', '079', '119', '118', '045', '055', '073', '230', '059', '038', '109', '046', '097', '091', '220', '077', '054', '071', '057', '159', '040', '042', '051', '165', '107', '087', '125', '240', '106', '104', '172', '096', '132', '162', '215', '094', '061', '047', '135', '147', '117', '056', '144', '190', '116', '064', '114', '058', '123', '145', '210', '225', '136', '133', '111', '102', '265', '186', '048', '195', '185', '066', '155', '214', '113', '081', '260', '196', '176', '143', '270', '175', '290', '228', '189', '128', '192', '138', '167', '052', '178', '121', '139', '036', '216', '242', '168',

In [20]:
df_gasto_energetico.loc[:,"VALOREST"] = df_gasto_energetico["VALOREST"].replace(-9, np.nan)

In [21]:
cols = ["NMIEMSD", "NMIEMHU", "NMIEMIN"]

for col in cols:
    vc = df_gasto_energetico[col].value_counts(dropna=False)
    pct = (df_gasto_energetico[col].value_counts(normalize=True, dropna=False) * 100).round(2)
    tabla = pd.DataFrame({"frecuencia": vc, "porcentaje_%": pct})
    print(f"\n=== Distribución de {col} ===")
    print(tabla)


=== Distribución de NMIEMSD ===
         frecuencia  porcentaje_%
NMIEMSD                          
0             19518         99.87
1                25          0.13
2                 1          0.01

=== Distribución de NMIEMHU ===
         frecuencia  porcentaje_%
NMIEMHU                          
0             19543         99.99
1                 1          0.01

=== Distribución de NMIEMIN ===
         frecuencia  porcentaje_%
NMIEMIN                          
0             19537         99.96
1                 5          0.03
3                 1          0.01
2                 1          0.01


In [22]:
columnas_a_eliminar += ["NMIEMSD", "NMIEMHU", "NMIEMIN"]

In [23]:
columnas_a_eliminar

['GASTOT',
 'GASTMON',
 'GASTNOM1',
 'GASTNOM2',
 'GASTNOM3',
 'GASTNOM4',
 'GASTOTGR01',
 'GASTOTGR02',
 'GASTOTGR03',
 'GASTOTGR04',
 'GASTOTGR05',
 'GASTOTGR06',
 'GASTOTGR07',
 'GASTOTGR08',
 'GASTOTGR09',
 'GASTOTGR10',
 'GASTOTGR11',
 'GASTOTGR12',
 'GASTOTGR13',
 'NUMERO',
 'ANOENC',
 'TAMANO',
 'IMPUTGAS',
 'NORDENSP',
 'NORDENCOSP',
 'NORDENP1SP',
 'NORDENP2SP',
 'CLAVE',
 'CLATEO',
 'FACTOR',
 'IMPEXACPSP',
 'INTERINPSP',
 'INTERIN',
 'NMIEMSD',
 'NMIEMHU',
 'NMIEMIN']

In [21]:
mask =  ~df_gasto_energetico.columns.isin(columnas_a_eliminar)
df_gasto_energetico2 =  df_gasto_energetico.loc[:,mask]
df_gasto_energetico2.to_csv("../data/processed/gasto_energetico_hogar2.csv", index=False)

with pd.option_context("display.max_columns", None, "display.width", None):
    display(df_gasto_energetico2.head(5))

df_gasto_energetico2.info()

,GASTO_SUMINISTROS,CCAA,NUTS1,CAPROV,TAMAMU,DENSIDAD,NMIEMB,NMIEM1,NMIEM2,NMIEM3,NMIEM4,NMIEM5,NMIEM6,NMIEM7,NMIEM8,NMIEM9,NMIEM10,NMIEM11,NMIEM12,NMIEM13,NUMACTI,NUMINACTI,NUMOCUP,NUMNOCUP,NUMESTU,NUMNOESTU,NNINOSD,NHIJOSD,UC1,UC2,PF2TEO,PF2RECO,TIPHOGAR1,TIPHOGAR7,TIPHOGAR8,EDADSP,SEXOSP,PAISNACSP,NACIONASP,SITURESSP,ESTADOCIVILSP,PAISPADRESP,PAISMADRESP,ESTUDIOSSP,ESTUDREDSP,SITUACTSP,SITUREDSP,JORNADASP,OCUSP,PERCEPSP,TRABAJO,OCUPA,OCUPARED,ACTESTB,ACTESTBRED,SITPROF,SECTOR,CONTRATO,TIPOCONT,SITSOCI,SITSOCIRE,REGTEN,TIPOEDIF,ZONARES,TIPOCASA,NHABIT,ANNOCON,SUPERF,AGUACALI,FUENAGUA,CALEF,FUENCALE,VALOREST,DISPOSIOV,CAPROP,CAJENA,PENSIOJUB,PENSIOINCAP,PENSIOTR,DESEM,OTRSUB,RENTAS,OTROIN,FUENPRIN,FUENPRINRED,IMPEXAC,NUMPERI
0,760.428414,Cataluña,Este,No,Municipio con 10.000 o más y menos de 20.000 h...,Zona intermedia,4,3,1,2,2,2,2,0,2,0,0,2,0,0,2,2,2,2,0,4,2,2,2.9,2.3,2,2,Pareja con dos hijos menores de 16 años,Pareja con dos hijos,Pareja con hijos menores de 16 años o adulto ...,45,Hombre,Resto de la Unión Europea (27 países),Sólo extranjera,Presente,Casado,Resto de la Unión Europea (27 países),Resto de la Unión Europea (27 países),"Grado de más de 240 ECTS, Licenciatura, Arquit...",Educación superior,Trabajando o con trabajo del que está ausente ...,Activo,Completa,Ocupado,Si,Si,"Empleados contables, administrativos y otros e...",Empleados de tipo administrativo y trabajadore...,Administración pública y defensa; seguridad so...,Servicios,Asalariado,Público,Con contrato,Indefinido,"Sector público. Trabajador no manual, excepto ...","Trabajador no manual, excepto agricultura",En propiedad sin préstamo o hipoteca en curso,Edificio con 10 o más viviendas,Urbana media,Casa media,5.0,Hace 25 o más años,85,Si,Electricidad,Si,Electricidad,150000.0,No,No,Si,No,No,No,No,No,Si,No,Trabajo por cuenta ajena,Trabajo por cuenta ajena,4000,2
1,1030.630587,Comunitat Valenciana,Este,Sí,Municipio de 100.000 habitantes o más,Zona densamente poblada,2,2,0,0,2,0,2,0,0,0,0,0,2,0,0,2,0,2,0,2,0,0,1.7,1.5,1,1,Pareja sin hijos teniendo al menos uno de los ...,Pareja sin hijos,Persona o pareja (al menos uno de los miembros...,75,Hombre,España,Sólo española,Presente,Casado,España,España,"Grado de 240 ECTS, Diplomatura, Arquitectura e...",Educación superior,"Jubilado/a, retirado/a anticipadamente",Inactivo,NaN,No ocupado,No consta,Si,Directores y gerentes,Directores y gerentes,Actividades financieras y de seguros,Servicios,Asalariado,Privado,Con contrato,Indefinido,"Jubilado/a, retirado/a anticipadamente",Retirados o jubilados,En propiedad sin préstamo o hipoteca en curso,Vivienda unifamiliar independiente,Urbana alta,Chalé o casa grande,7.0,Hace 25 o más años,280,Si,Gas natural,Si,Gas natural,400000.0,No,No,No,Si,No,No,No,No,No,No,Pensiones de jubilación,"Pensiones, subsidios y otras prestaciones e in...",7614,2
2,1265.599996,Galicia,Noroeste,Sí,Municipio de 100.000 habitantes o más,Zona densamente poblada,2,2,0,0,2,0,2,0,0,0,1,1,0,0,2,0,1,1,0,2,0,0,1.7,1.5,1,1,Otros hogares,Otro tipo de hogares,Otros hogares,39,Mujer,Resto del mundo,Española y extranjera,Presente,Soltero,Resto del mundo,Resto del mundo,"Bachiller, BUP, COU, Bachiller Superior, FP de...",Segunda etapa de Educación secundaria,Trabajando o con trabajo del que está ausente ...,Activo,Completa,Ocupado,No consta,Si,"Trabajadores de los servicios de restauración,...",Empleados de tipo administrativo y trabajadore...,Hostelería,Servicios,Asalariado,Privado,Con contrato,Indefinido,"Sector privado. Trabajador manual, excepto agr...","Trabajador manual, excepto agricultura",Alquiler,Edificio con menos de 10 viviendas,Urbana media,Casa media,5.0,Hace 25 o más años,90,Si,Gas licuado,No,NaN,NaN,No,No,Si,No,No,No,Si,No,No,No,Trabajo por cuenta ajena,Trabajo por cuenta ajena,2213,2
3,1068.580361,Cataluña,Este,No,Municipio de 100.000 habitantes o más,Zona densamente poblada,4,2,2,2,2,2,2,1,1,0,1,1,0,0,1,3,1,3,0,4,2,2,2.7,2.1,1,1,Pareja con dos hijos menores de 16 años,Pareja con dos hijos,Pareja con h

<class 'pandas.DataFrame'>
RangeIndex: 19544 entries, 0 to 19543
Data columns (total 87 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GASTO_SUMINISTROS  19544 non-null  float64
 1   CCAA               19544 non-null  str    
 2   NUTS1              19544 non-null  str    
 3   CAPROV             19544 non-null  str    
 4   TAMAMU             19544 non-null  str    
 5   DENSIDAD           19544 non-null  str    
 6   NMIEMB             19544 non-null  int64  
 7   NMIEM1             19544 non-null  int64  
 8   NMIEM2             19544 non-null  int64  
 9   NMIEM3             19544 non-null  int64  
 10  NMIEM4             19544 non-null  int64  
 11  NMIEM5             19544 non-null  int64  
 12  NMIEM6             19544 non-null  int64  
 13  NMIEM7             19544 non-null  int64  
 14  NMIEM8             19544 non-null  int64  
 15  NMIEM9             19544 non-null  int64  
 16  NMIEM10            19544 non-null